# Notebook 14 — Final Evaluation & Comparison

**Tujuan:** Evaluasi akhir dan perbandingan komprehensif seluruh pendekatan optimasi yang telah dieksperimenkan pada sistem photo clustering berbasis HDBSCAN.

---

## Ringkasan Eksperimen

| NB | Pendekatan | Inti Ide |
|----|-----------|---------|
| NB8 | **Baseline** | HDBSCAN + Correlation distance, hyperparameter tuning |
| NB9 | **UMAP + HDBSCAN** | Dimensionality reduction sebelum clustering |
| NB10 | **Angular Distance** | Ganti metrik jarak + soft clustering |
| NB12 | **CGA** | Gaussian augmentation pada minority cluster |
| NB13 | **NT-CGA** | Gaussian augmentation pada noise singletons (probable) |

---

**Metrik evaluasi:**
- **Coverage Rate** (↑): persentase data yang berhasil di-cluster
- **Silhouette Score** (↑): kualitas kekompakan & separabilitas cluster
- **Noise Ratio** (↓): persentase data yang dicap noise
- **N Clusters**: jumlah cluster yang terbentuk

## Cell 1 — Imports & Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from math import pi

# ── Warna per metode ─────────────────────────────────────────────────────────
COLORS = {
    'Baseline':   '#636EFA',
    'UMAP':       '#00CC96',
    'Angular':    '#EF553B',
    'CGA':        '#AB63FA',
    'NT-CGA':     '#FFA15A',
}

RESULTS_DIR = '/content/drive/MyDrive/OTW S.KOM/Results'
PLOTS_DIR   = os.path.join(RESULTS_DIR, 'Plots')
os.makedirs(PLOTS_DIR, exist_ok=True)

print('✅ Imports OK')

## Cell 2 — Definisi Data Hasil Eksperimen

Data diisi dari output notebook 8–13. PKL di-load jika tersedia; fallback ke hardcoded values.

In [ ]:
# ── Hardcoded best results dari masing-masing notebook ───────────────────────
# Sumber: output cell summary tiap notebook

TOTAL_DATA = 12715

results_hardcoded = [
    {
        'method':      'Baseline',
        'notebook':    'NB8',
        'description': 'HDBSCAN + Correlation (mcs=15, ms=70)',
        'clusters':    54,
        'noise':       5565,
        'coverage':    56.2,
        'silhouette':  0.4530,
        'approach':    'Distance Metric Tuning',
    },
    {
        'method':      'UMAP',
        'notebook':    'NB9',
        'description': 'UMAP (cosine, n_comp=30, n_neigh=30) + HDBSCAN (mcs=20, ms=20)',
        'clusters':    95,
        'noise':       90,
        'coverage':    99.3,
        'silhouette':  0.9041,
        'approach':    'Dimensionality Reduction',
    },
    {
        'method':      'Angular',
        'notebook':    'NB10',
        'description': 'Angular Distance / L2-norm Euclidean (mcs=10, ms=20)',
        'clusters':    90,
        'noise':       2526,
        'coverage':    80.1,
        'silhouette':  0.2573,
        'approach':    'Distance Metric Change',
    },
    {
        'method':      'CGA',
        'notebook':    'NB12',
        'description': 'Cluster-Guided Gaussian Augmentation (α=0.5, n_syn=10)',
        'clusters':    54,
        'noise':       5544,
        'coverage':    56.4,
        'silhouette':  0.4547,
        'approach':    'Data Augmentation',
    },
    {
        'method':      'NT-CGA',
        'notebook':    'NB13',
        'description': 'Noise-Targeted CGA (α=1.0, n_syn=5, mcs_recluster=7)',
        'clusters':    348,
        'noise':       2479,
        'coverage':    80.5,
        'silhouette':  0.3005,
        'approach':    'Noise-Targeted Augmentation',
    },
]

df = pd.DataFrame(results_hardcoded)
df['noise_ratio']    = df['noise'] / TOTAL_DATA * 100
df['clustered']      = TOTAL_DATA - df['noise']
df['delta_coverage'] = df['coverage'] - df.loc[df['method']=='Baseline','coverage'].values[0]
df['delta_sil']      = df['silhouette'] - df.loc[df['method']=='Baseline','silhouette'].values[0]

print('\n✅ Data hasil eksperimen berhasil dimuat')
print(f'   Total data: {TOTAL_DATA:,}')
print(f'   Jumlah metode: {len(df)}')

## Cell 3 — Master Comparison Table

In [ ]:
def fmt_delta(val, is_pct=False):
    suffix = '%' if is_pct else ''
    if val > 0:
        return f'+{val:.2f}{suffix}'
    elif val < 0:
        return f'{val:.2f}{suffix}'
    else:
        return '—'

print('=' * 105)
print('  MASTER COMPARISON TABLE — SEMUA METODE OPTIMASI HDBSCAN')
print('=' * 105)
header = f"{'Method':<12} {'NB':<6} {'Approach':<30} {'Clusters':>9} {'Noise':>7} {'Coverage':>10} {'Silhouette':>12} {'ΔCoverage':>11} {'ΔSil':>9}"
print(header)
print('─' * 105)

for _, row in df.iterrows():
    marker = '◀ BEST COV' if row['coverage'] == df['coverage'].max() else (
             '◀ BEST SIL' if row['silhouette'] == df['silhouette'].max() else '')
    print(
        f"{row['method']:<12} {row['notebook']:<6} {row['approach']:<30} "
        f"{row['clusters']:>9,} {row['noise']:>7,} {row['coverage']:>9.1f}% "
        f"{row['silhouette']:>12.4f} "
        f"{fmt_delta(row['delta_coverage'], True):>11} "
        f"{fmt_delta(row['delta_sil']):>9}  {marker}"
    )

print('─' * 105)
print()
print('Catatan:')
print('  Coverage = % data berhasil di-cluster  |  Silhouette = kualitas cluster (0–1)')
print('  ΔCoverage & ΔSil = delta terhadap Baseline')

## Cell 4 — Visualisasi: Bar Chart Perbandingan Metrik

In [ ]:
methods = df['method'].tolist()
colors  = [COLORS[m] for m in methods]
x       = np.arange(len(methods))
bar_w   = 0.6

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Perbandingan Metode Optimasi HDBSCAN — Semua Metrik Utama',
             fontsize=15, fontweight='bold', y=1.01)

# ── 1. Coverage Rate ─────────────────────────────────────────────────────────
ax = axes[0, 0]
bars = ax.bar(x, df['coverage'], color=colors, width=bar_w, edgecolor='white', linewidth=0.8)
ax.axhline(df.loc[df['method']=='Baseline','coverage'].values[0],
           color='grey', linestyle='--', linewidth=1.2, label='Baseline')
for bar, val in zip(bars, df['coverage']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9.5, fontweight='bold')
ax.set_title('Coverage Rate (↑ lebih baik)', fontsize=12, fontweight='bold')
ax.set_ylabel('Coverage (%)')
ax.set_ylim(0, 115)
ax.set_xticks(x); ax.set_xticklabels(methods, fontsize=10)
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

# ── 2. Silhouette Score ───────────────────────────────────────────────────────
ax = axes[0, 1]
bars = ax.bar(x, df['silhouette'], color=colors, width=bar_w, edgecolor='white', linewidth=0.8)
ax.axhline(df.loc[df['method']=='Baseline','silhouette'].values[0],
           color='grey', linestyle='--', linewidth=1.2, label='Baseline')
for bar, val in zip(bars, df['silhouette']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9.5, fontweight='bold')
ax.set_title('Silhouette Score (↑ lebih baik)', fontsize=12, fontweight='bold')
ax.set_ylabel('Silhouette Score')
ax.set_ylim(0, 1.05)
ax.set_xticks(x); ax.set_xticklabels(methods, fontsize=10)
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

# ── 3. Noise Ratio ────────────────────────────────────────────────────────────
ax = axes[1, 0]
bars = ax.bar(x, df['noise_ratio'], color=colors, width=bar_w, edgecolor='white', linewidth=0.8)
ax.axhline(df.loc[df['method']=='Baseline','noise_ratio'].values[0],
           color='grey', linestyle='--', linewidth=1.2, label='Baseline')
for bar, val in zip(bars, df['noise_ratio']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9.5, fontweight='bold')
ax.set_title('Noise Ratio (↓ lebih baik)', fontsize=12, fontweight='bold')
ax.set_ylabel('Noise Ratio (%)')
ax.set_ylim(0, 55)
ax.set_xticks(x); ax.set_xticklabels(methods, fontsize=10)
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

# ── 4. Jumlah Cluster ─────────────────────────────────────────────────────────
ax = axes[1, 1]
bars = ax.bar(x, df['clusters'], color=colors, width=bar_w, edgecolor='white', linewidth=0.8)
ax.axhline(df.loc[df['method']=='Baseline','clusters'].values[0],
           color='grey', linestyle='--', linewidth=1.2, label='Baseline')
for bar, val in zip(bars, df['clusters']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            f'{val:,}', ha='center', va='bottom', fontsize=9.5, fontweight='bold')
ax.set_title('Jumlah Cluster', fontsize=12, fontweight='bold')
ax.set_ylabel('N Clusters')
ax.set_xticks(x); ax.set_xticklabels(methods, fontsize=10)
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
path1 = os.path.join(PLOTS_DIR, 'nb14_01_bar_comparison.png')
plt.savefig(path1, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Plot disimpan: {path1}')

## Cell 5 — Visualisasi: Coverage vs Silhouette Trade-off

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Trade-off Analysis: Coverage vs Silhouette', fontsize=14, fontweight='bold')

# ── Scatter: Coverage vs Silhouette ──────────────────────────────────────────
ax = axes[0]
for _, row in df.iterrows():
    c = COLORS[row['method']]
    ax.scatter(row['coverage'], row['silhouette'],
               color=c, s=200, zorder=5, edgecolors='white', linewidth=1.5)
    offset_x = 0.5
    offset_y = 0.015
    if row['method'] == 'CGA':
        offset_x = -5; offset_y = 0.015
    if row['method'] == 'NT-CGA':
        offset_x = 0.5; offset_y = -0.03
    ax.annotate(row['method'],
                (row['coverage'], row['silhouette']),
                textcoords='offset points',
                xytext=(offset_x*10, offset_y*100),
                fontsize=10, fontweight='bold', color=c)

# Annotate ideal region
ax.annotate('Ideal\n(high cov,\nhigh sil)', xy=(95, 0.85),
            fontsize=9, color='green', alpha=0.7,
            arrowprops=dict(arrowstyle='->', color='green', alpha=0.5),
            xytext=(75, 0.75))

ax.set_xlabel('Coverage Rate (%)', fontsize=11)
ax.set_ylabel('Silhouette Score', fontsize=11)
ax.set_title('Coverage vs Silhouette per Metode', fontsize=11, fontweight='bold')
ax.set_xlim(45, 105)
ax.set_ylim(0.15, 1.0)
ax.grid(alpha=0.3)

# ── Delta bar chart ────────────────────────────────────────────────────────────
ax2 = axes[1]
non_base = df[df['method'] != 'Baseline'].copy()
x2 = np.arange(len(non_base))
w  = 0.35

bar1 = ax2.bar(x2 - w/2, non_base['delta_coverage'], width=w,
               label='ΔCoverage (%)', color=[COLORS[m] for m in non_base['method']],
               alpha=0.85, edgecolor='white')
bar2 = ax2.bar(x2 + w/2, non_base['delta_sil'] * 100, width=w,
               label='ΔSilhouette ×100', color=[COLORS[m] for m in non_base['method']],
               alpha=0.45, edgecolor='white', hatch='//')

for bar, val in zip(bar1, non_base['delta_coverage']):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+(0.4 if val>=0 else -2),
             f'+{val:.1f}%', ha='center', va='bottom', fontsize=8.5, fontweight='bold')
for bar, val in zip(bar2, non_base['delta_sil']):
    lbl = f'+{val:.4f}' if val>=0 else f'{val:.4f}'
    ypos = bar.get_height() if val>=0 else bar.get_height()-3
    ax2.text(bar.get_x()+bar.get_width()/2, ypos+(0.5 if val>=0 else -1.5),
             lbl, ha='center', va='bottom', fontsize=8.5)

ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_xticks(x2)
ax2.set_xticklabels(non_base['method'].tolist(), fontsize=10)
ax2.set_ylabel('Delta vs Baseline')
ax2.set_title('Delta per Metode vs Baseline', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
path2 = os.path.join(PLOTS_DIR, 'nb14_02_tradeoff_delta.png')
plt.savefig(path2, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Plot disimpan: {path2}')

## Cell 6 — Visualisasi: Radar Chart Multi-Metrik

In [ ]:
# Normalisasi ke [0,1] untuk radar (silhouette langsung, coverage/100,
# noise_ratio dibalik (1 - noise/100), clusters dinormalisasi ke max)

radar_categories = ['Coverage\n(%/100)', 'Silhouette', '1 - Noise\nRatio', 'Cluster\nDiversity']
N = len(radar_categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # close the loop

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.set_theta_offset(pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_categories, fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25','0.50','0.75','1.0'], fontsize=8)
ax.grid(alpha=0.4)

max_clusters = df['clusters'].max()

for _, row in df.iterrows():
    vals = [
        row['coverage'] / 100,
        row['silhouette'],
        1 - row['noise_ratio'] / 100,
        row['clusters'] / max_clusters,
    ]
    vals += vals[:1]
    c = COLORS[row['method']]
    ax.plot(angles, vals, color=c, linewidth=2, linestyle='solid', label=row['method'])
    ax.fill(angles, vals, color=c, alpha=0.08)

ax.set_title('Radar Chart — Perbandingan Multi-Metrik\n(semua dinormalisasi ke [0,1])',
             fontsize=12, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10)

path3 = os.path.join(PLOTS_DIR, 'nb14_03_radar.png')
plt.savefig(path3, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Plot disimpan: {path3}')

## Cell 7 — Stacked Bar: Komposisi Data per Metode

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(df))
bar_w = 0.55

p1 = ax.bar(x, df['clustered'], bar_w, label='Berhasil Di-cluster',
            color=[COLORS[m] for m in df['method']], alpha=0.85, edgecolor='white')
p2 = ax.bar(x, df['noise'], bar_w, bottom=df['clustered'], label='Noise (unclustered)',
            color='#D3D3D3', alpha=0.7, edgecolor='white')

# Label dalam bar
for i, (_, row) in enumerate(df.iterrows()):
    ax.text(i, row['clustered']/2, f"{row['clustered']:,}\n({row['coverage']:.1f}%)",
            ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    ax.text(i, row['clustered'] + row['noise']/2, f"{row['noise']:,}\n({row['noise_ratio']:.1f}%)",
            ha='center', va='center', fontsize=8.5, color='#555')

ax.axhline(TOTAL_DATA, color='black', linestyle=':', linewidth=1, label=f'Total data ({TOTAL_DATA:,})')
ax.set_xticks(x)
ax.set_xticklabels([f"{row['method']}\n({row['notebook']})" for _, row in df.iterrows()], fontsize=10)
ax.set_ylabel('Jumlah Data Points', fontsize=11)
ax.set_title('Komposisi Data: Clustered vs Noise — Tiap Metode', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, TOTAL_DATA * 1.08)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
path4 = os.path.join(PLOTS_DIR, 'nb14_04_stacked_composition.png')
plt.savefig(path4, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Plot disimpan: {path4}')

## Cell 8 — Analisis Temuan & Trade-off

In [ ]:
print('=' * 80)
print('  ANALISIS TEMUAN UTAMA')
print('=' * 80)

best_cov = df.loc[df['coverage'].idxmax()]
best_sil = df.loc[df['silhouette'].idxmax()]
baseline = df.loc[df['method']=='Baseline'].iloc[0]

print(f"""
┌──────────────────────────────────────────────────────────────────────────────┐
│  1. UMAP + HDBSCAN (NB9) — TERTINGGI COVERAGE & SILHOUETTE                  │
│     Coverage : {best_cov['coverage']:.1f}%  (Δ={best_cov['delta_coverage']:+.1f}% vs baseline)                       │
│     Silhouette: {best_cov['silhouette']:.4f} (Δ={best_cov['delta_sil']:+.4f} vs baseline)                   │
│                                                                              │
│     UMAP mereduksi dimensi 512→30, mendekatkan embedding yang semantically   │
│     mirip. Noise rate turun drastis (43.8% → 0.7%), tetapi silhouette       │
│     yang tinggi (0.9041) perlu dicek: apakah merupakan artefak dari          │
│     kompresi UMAP, bukan refleksi kualitas cluster yang sesungguhnya.        │
├──────────────────────────────────────────────────────────────────────────────┤
│  2. ANGULAR & NT-CGA — COVERAGE TINGGI DENGAN SILHOUETTE LEBIH RENDAH       │
│     Angular: Coverage 80.1%, Sil=0.2573 (Δ={df.loc[df['method']=='Angular','delta_sil'].values[0]:+.4f})                     │
│     NT-CGA : Coverage 80.5%, Sil=0.3005 (Δ={df.loc[df['method']=='NT-CGA','delta_sil'].values[0]:+.4f})                     │
│                                                                              │
│     Keduanya berhasil meningkatkan coverage ~80%, tetapi silhouette turun.   │
│     NT-CGA membentuk 348 cluster (vs 54 baseline): cluster baru cenderung   │
│     kecil dan less cohesive — konsekuensi dari augmentasi Gaussian.          │
├──────────────────────────────────────────────────────────────────────────────┤
│  3. CGA (NB12) — KONFIRMASI HIPOTESIS SALAH TARGET                          │
│     Coverage hanya +0.2% (vs +24.3% NT-CGA) karena CGA menyasar minority   │
│     cluster bukan noise singletons. Ini membuktikan bahwa masalah utama      │
│     adalah noise singletons, bukan minority clusters.                        │
├──────────────────────────────────────────────────────────────────────────────┤
│  4. TRADE-OFF FUNDAMENTAL: COVERAGE vs SILHOUETTE QUALITY                   │
│     Setiap metode yang meningkatkan coverage drastis mengalami trade-off:    │
│     • Angular  : +23.9% coverage, −0.1957 silhouette                        │
│     • NT-CGA   : +24.3% coverage, −0.1525 silhouette                        │
│     Pengecualian: UMAP (+43.1% coverage, +0.4511 sil) — tapi perlu          │
│     validasi apakah silhouette UMAP comparable dengan raw embedding.        │
└──────────────────────────────────────────────────────────────────────────────┘
""")

print('=' * 80)
print('  ROOT CAUSE ANALYSIS — NOISE PROBLEM IN HDBSCAN')
print('=' * 80)
print(f"""
  Dari NB11 (Noise Composition Analysis), noise {int(baseline['noise']):,} titik ({baseline['noise_ratio']:.1f}%) terdiri dari:
  • Probable Singletons  : 1,395 (25.1%) — DNCN < 16.98 = foto wajah jarang
  • Borderline           : 2,779 (49.9%) — ambigu, dekat cluster
  • Probable Outliers    : 1,391 (25.0%) — benar-benar noise

  Implikasi:
  • ~25% noise adalah wajah valid yang tidak muncul cukup sering (probable singletons)
  • HDBSCAN menyamakan mereka dengan outlier murni karena min_cluster_size
  • Ini adalah keterbatasan density-based clustering yang diakui di literatur
    (Campello et al., 2019; Bushra & Yi, 2021)
""")

## Cell 9 — Kesimpulan Akhir & Rekomendasi

In [ ]:
print('=' * 80)
print('  KESIMPULAN AKHIR — OPTIMASI SISTEM PHOTO CLUSTERING BERBASIS HDBSCAN')
print('=' * 80)

print("""
┌──────────────────────────────────────────────────────────────────────────────┐
│                    RANGKUMAN EKSPERIMEN (NB8 – NB13)                        │
├──────────────────────────────┬──────────────┬────────────┬───────────────────┤
│ Metode                       │ Coverage     │ Silhouette │ Highlight         │
├──────────────────────────────┼──────────────┼────────────┼───────────────────┤
│ Baseline (Correlation)       │ 56.2%        │ 0.4530     │ Referensi         │
│ UMAP + HDBSCAN (NB9)         │ 99.3% (+43.1%)│ 0.9041    │ Best semua metrik │
│ Angular Distance (NB10)      │ 80.1% (+23.9%)│ 0.2573    │ Coverage tinggi   │
│ CGA Augmentation (NB12)      │ 56.4%  (+0.2%)│ 0.4547    │ Target salah      │
│ NT-CGA (NB13)                │ 80.5% (+24.3%)│ 0.3005    │ Noise recovery    │
└──────────────────────────────┴──────────────┴────────────┴───────────────────┘

TEMUAN KUNCI:
  1. UMAP adalah pendekatan terkuat secara metrik, namun perlu validasi
     apakah silhouette pada ruang UMAP sebanding dengan ruang embedding asli.

  2. NT-CGA membuktikan bahwa noise singletons (25% dari total noise) adalah
     target yang tepat. Peningkatan +24.3% coverage signifikan.

  3. CGA standar (NB12) membuktikan: augmentasi pada minority cluster tidak
     efektif karena masalah bukan di sana.

  4. Trade-off Coverage–Silhouette adalah temuan teoritis yang relevan:
     optimasi coverage tanpa menjaga cluster cohesion menghasilkan fragmentasi.

LIMITASI:
  • Silhouette NT-CGA dihitung pada ruang augmented — mungkin bias.
  • Tidak ada ground truth label untuk validasi kualitatif cluster.
  • Cluster NT-CGA (348 cluster) belum divalidasi apakah merupakan
    identitas wajah yang benar-benar berbeda.

REKOMENDASI EKSPERIMEN LANJUTAN:
  • NB15: KNN-based noise assignment (post-processing HDBSCAN)
    → Langsung assign noise point ke cluster terdekat berdasarkan k-NN
    → Referensi: Patel & Rathore (2025) — PDBSCAN
  • NB16: HRSB-SMOTE style augmentation
    → Gunakan grade of membership HDBSCAN untuk filter noise sebelum augmentasi
    → Referensi: Xue et al. (2024) — HRSB-SMOTE
""")

print('=' * 80)

## Cell 10 — Simpan Hasil Final

In [ ]:
# Simpan dataframe perbandingan
csv_path = os.path.join(RESULTS_DIR, 'notebook14_final_comparison.csv')
pkl_path = os.path.join(RESULTS_DIR, 'notebook14_final_comparison.pkl')

df.to_csv(csv_path, index=False)
df.to_pickle(pkl_path)

print(f'💾 CSV disimpan : {csv_path}')
print(f'💾 PKL disimpan : {pkl_path}')
print()
print('=' * 70)
print('✅ Notebook 14 selesai.')
print('   Final Evaluation & Comparison telah terdokumentasi.')
print()
print('   Eksperimen lanjutan yang direncanakan:')
print('   • NB15: KNN-based Noise Assignment (PDBSCAN-style)')
print('   • NB16: Grade-of-Membership Augmentation (HRSB-SMOTE style)')
print('=' * 70)